# Hardware repetition-code suppression protocol

Companion notebook to **Lattice Atlas**. This file is an unexecuted, credential-gated protocol; it is not a committed hardware result.

1. **Browser Lab** — an ideal-check, i.i.d. data-Pauli toy
2. **Stim + PyMatching** (`first-threshold-curve.ipynb`) — a generated circuit/noise/decoder simulation
3. **This notebook** — a repetition-code hardware test whose outcome may be suppressive, non-suppressive, or inconclusive

The circuit is a **bit-flip repetition-code memory** with only a final data measurement. It protects one error type and has no repeated ancilla syndrome extraction, so it is not a surface-code or fault-tolerant-memory experiment. Provider access, pricing, quotas, and available backends change; check the current service terms before running.

In [ ]:
%pip install -q qiskit qiskit-aer qiskit-ibm-runtime matplotlib numpy

## 1 · The experiment

Hold logical |1⟩ = |1…1⟩ on a chain of n qubits through a stretch of idle time (built from X-gate pairs — the barriers stop the transpiler from cancelling them), then measure and **decode by majority vote**. Amplitude damping (T1 decay) flips qubits toward |0⟩; the vote outvotes minority flips. A logical error = the majority flipped.

Prediction from the theory you learned on the site: logical error should **fall as n grows** — same scaling law as the Lab's chart, on hardware.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit

def repetition_memory(n_qubits: int, idle_pairs: int) -> QuantumCircuit:
    qc = QuantumCircuit(n_qubits, n_qubits)
    qc.x(range(n_qubits))
    qc.barrier()
    for _ in range(idle_pairs):
        qc.x(range(n_qubits))
        qc.barrier()
        qc.x(range(n_qubits))
        qc.barrier()
    qc.measure(range(n_qubits), range(n_qubits))
    return qc

def logical_error_rate(counts: dict, n_qubits: int):
    shots = sum(counts.values())
    fails = sum(c for bits, c in counts.items() if bits.count('1') <= n_qubits // 2)
    return fails / shots, shots

DISTANCES = [3, 5, 7]
IDLE_PAIRS = 24
SHOTS = 4000  # gentle on your free minutes; raise for tighter error bars

## 2 · Dry run on the local simulator first

This dry run attaches one illustrative thermal-relaxation channel to the X operations. It tests the notebook's counting path only; it is not a calibrated backend model and cannot validate credentials, transpilation, queues, or hardware behavior.

In [ ]:
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, thermal_relaxation_error

noise = NoiseModel()
noise.add_all_qubit_quantum_error(thermal_relaxation_error(300.0, 200.0, 1.0), ['x'])
sim = AerSimulator(noise_model=noise)

sim_rates = {}
for n in DISTANCES:
    counts = sim.run(repetition_memory(n, IDLE_PAIRS), shots=20_000).result().get_counts()
    rate, shots = logical_error_rate(counts, n)
    sim_rates[n] = rate
    print(f'd={n}: simulated logical error {rate:.4f}')
assert sim_rates[5] < sim_rates[3] and sim_rates[7] < sim_rates[5]
print('suppression confirmed on simulator — safe to spend hardware minutes')

## 3 · The real thing

Configure a provider account using its current instructions. Never commit a token. Inspect the selected backend and transpiled circuits before submission; `least_busy` and a preset pass manager do not guarantee comparable physical support or noise across distances.

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.transpiler import generate_preset_pass_manager

# First time only — then you can delete this line:
# QiskitRuntimeService.save_account(token='YOUR_TOKEN', set_as_default=True)

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)
print(f'running on {backend.name} ({backend.num_qubits} qubits)')

pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
isa_circuits = [pm.run(repetition_memory(n, IDLE_PAIRS)) for n in DISTANCES]

sampler = SamplerV2(mode=backend)
job = sampler.run(isa_circuits, shots=SHOTS)
print(f'job id: {job.job_id()} — waiting for the quantum computer…')
result = job.result()

In [ ]:
import importlib.metadata as metadata
import json
import matplotlib.pyplot as plt

def wilson_interval(failures, shots, z=1.96):
    phat = failures / shots
    denom = 1 + z*z/shots
    center = (phat + z*z/(2*shots)) / denom
    half = z * np.sqrt(phat*(1-phat)/shots + z*z/(4*shots*shots)) / denom
    return max(0.0, center-half), min(1.0, center+half)

hw_rates, hw_intervals, hw_counts = {}, {}, {}
for n, res in zip(DISTANCES, result):
    counts = res.data.c.get_counts()
    rate, shots = logical_error_rate(counts, n)
    failures = round(rate * shots)
    hw_rates[n] = rate
    hw_intervals[n] = wilson_interval(failures, shots)
    hw_counts[n] = counts
    lo, hi = hw_intervals[n]
    print(f'd={n}: {failures}/{shots} logical failures; rate={rate:.5f}, Wilson 95%=[{lo:.5f}, {hi:.5f}]')

strict_suppression = (hw_intervals[5][1] < hw_intervals[3][0]
                      and hw_intervals[7][1] < hw_intervals[5][0])
clear_reversal = (hw_intervals[5][0] > hw_intervals[3][1]
                  or hw_intervals[7][0] > hw_intervals[5][1])
if strict_suppression:
    evidence_status = 'SUPPRESSIVE: separated 95% intervals for d=3 > d=5 > d=7'
elif clear_reversal:
    evidence_status = 'NON-SUPPRESSIVE: at least one larger distance is clearly worse'
else:
    evidence_status = 'INCONCLUSIVE: intervals overlap; collect more shots or revise the experiment'
print(evidence_status)

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.errorbar(DISTANCES, [sim_rates[n] for n in DISTANCES], marker='s', ls='--',
            color='#0891B2', label='simulator (relaxation model)')
hardware_y = [hw_rates[n] if hw_rates[n] > 0 else hw_intervals[n][1] for n in DISTANCES]
hardware_lower = [y - hw_intervals[n][0] if hw_rates[n] > 0 else y/2
                  for n, y in zip(DISTANCES, hardware_y)]
hardware_upper = [hw_intervals[n][1] - y if hw_rates[n] > 0 else 0
                  for n, y in zip(DISTANCES, hardware_y)]
ax.errorbar(DISTANCES, hardware_y, yerr=[hardware_lower, hardware_upper],
            uplims=[hw_rates[n] == 0 for n in DISTANCES], marker='o',
            color='#D97706', capsize=4, label=f'REAL HARDWARE ({backend.name})')
ax.set_yscale('log')
ax.set_xticks(DISTANCES)
ax.set_xlabel('code distance (chain length)')
ax.set_ylabel('logical error rate')
ax.set_title(f'Repetition-code hardware test: {evidence_status.split(":")[0].lower()}')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

if hw_rates[5] > 0 and hw_rates[7] > 0:
    print(f'Λ(3→5) = {hw_rates[3]/hw_rates[5]:.1f} · Λ(5→7) = {hw_rates[5]/hw_rates[7]:.1f}')
print('Do not compare these repetition-code point ratios directly with Willow surface-code Lambda.')

receipt = {
    'evidence_status': evidence_status,
    'provider': 'IBM Quantum',
    'backend': str(backend.name),
    'job_id': job.job_id(),
    'shots_requested_per_circuit': SHOTS,
    'distances': DISTANCES,
    'idle_pairs': IDLE_PAIRS,
    'counts_by_distance': {str(n): hw_counts[n] for n in DISTANCES},
    'wilson_95': {str(n): hw_intervals[n] for n in DISTANCES},
    'versions': {name: metadata.version(name) for name in
                 ['qiskit', 'qiskit-ibm-runtime', 'numpy']},
}
print(json.dumps(receipt, indent=2))
print('Save this receipt with the executed notebook before making a hardware-evidence claim.')

## 4 · Interpret the run without forcing a success story

The code cell above reports **SUPPRESSIVE**, **NON-SUPPRESSIVE**, or **INCONCLUSIVE** from the measured Wilson intervals. Do not replace that result with an unconditional claim. Treat it as hardware evidence only after saving the executed outputs and receipt (backend, job ID, versions, raw counts, shots, circuit settings, and intervals).

**Honest gaps between this and full QEC** (each is a topic on the site):
- The repetition code protects only against bit flips — a surface code protects both X and Z (*quantum codes basics*).
- We measured once at the end; fault-tolerant memories repeatedly extract checks and decode measurement faults (*syndrome extraction*; the Stim notebook demonstrates one simulation model).
- Majority vote is the decoder here; surface codes need matching (*decoding/MWPM* — play Decoder Duel to feel why).

### Keep going
- Rerun at different `IDLE_PAIRS` — find where suppression breaks down as noise accumulates.
- Try `n = 9, 11` — does Λ hold?
- Advanced: if the selected backend currently supports the needed dynamic circuits, add ancilla checks and declare the resulting fault model.
- For Willow context, use Google's published memory data or the public `willow_pink` Cirq QVM; the QVM is classical simulation, not hardware access.